
# 03 — LoRA / PEFT, from Scratch

**Goal:** implement LoRA (Low-Rank Adaptation), verify (a) it's a no-op at initialization, (b) it
merges back into a plain linear layer exactly, and be fluent on why it's structured the way it
is, when it does/doesn't cost you at inference, and how it relates to QLoRA. This is one of the
most frequently-cited coding exercises in current LLM interviews — "implement LoRA" comes up
constantly for anyone who might touch fine-tuning.

Structure: **Lesson → Implementation → Quiz → Final Answers & Explanations.**



## 1. Lesson

### 1.1 The problem

Full fine-tuning of a large pretrained model updates *every* weight matrix, which means: (a) you
need optimizer state (e.g. Adam's two extra buffers) for every parameter, multiplying training
memory several-fold over the model's own size; (b) you end up with a full separate copy of the
model's weights per downstream task/adapter, which is enormously wasteful if you want to serve
many fine-tuned variants of the same base model.

### 1.2 The low-rank hypothesis and the LoRA formula

LoRA's empirical premise: the *update* to a weight matrix needed to adapt a pretrained model to a
new task tends to have low "intrinsic rank" — i.e. even though $W \in \mathbb{R}^{d_{out}\times
d_{in}}$ is a big matrix, the useful change $\Delta W$ during fine-tuning can be well
approximated by a low-rank product. Concretely, LoRA freezes the pretrained $W$ entirely and
learns a *replacement* update
$$
W' = W + \Delta W = W + \frac{\alpha}{r} BA
$$
where $A \in \mathbb{R}^{r \times d_{in}}$, $B \in \mathbb{R}^{d_{out} \times r}$, and $r \ll
\min(d_{in}, d_{out})$ is the chosen rank. $\alpha$ is a fixed scaling constant (often set equal
to $r$, so $\alpha/r = 1$ by default, then tuned as a hyperparameter independent of $r$ so you
can change rank without automatically changing the effective learning-rate-like scale of the
update). Only $A$ and $B$ are trained; $W$ is frozen.

### 1.3 Zero-init, again

$A$ is randomly initialized (small values), but $B$ is initialized to **exactly zero**. This is
the same trick as Flamingo's gated cross-attention (notebook 1): at the start of training,
$BA = 0$, so $W' = W$ exactly — the adapted model is *identical* to the pretrained base model
until training actually moves $B$ away from zero. This guarantees fine-tuning starts from the
pretrained model's own behavior rather than an immediate random perturbation.

### 1.4 Where LoRA is applied, and why

LoRA is usually applied to the attention projection matrices (commonly $W_Q$ and $W_V$, and often
$W_K$, $W_O$ too) rather than the large FFN matrices, based on empirical findings that adapting
attention's query/value projections captures most of the benefit at much lower parameter cost
than adapting everything, though modern recipes vary (some apply LoRA broadly, including FFNs,
when compute allows). This is a "well-established practical default, not a hard law" kind of
fact — worth stating with that caveat in an interview.

### 1.5 Merging for inference, and multi-adapter serving

Because $W' = W + \frac{\alpha}{r}BA$ is just matrix addition, once training is done you can
**merge**: compute $W_{merged} = W + \frac{\alpha}{r}BA$ once and load it as a plain linear layer
— inference then costs *exactly* the same as the unmodified base model, with zero added latency
or extra parameters at serving time. The catch: merging commits you to *one* adapter baked into
the weights. If you want to serve many different LoRA adapters (e.g. one per customer/task)
against a shared frozen base model, you *can't* merge per-request without paying the cost of
either storing a full merged copy per adapter or re-merging on every request switch — production
multi-adapter serving systems instead keep adapters unmerged and apply the small extra $BA$ term
per-request (batched cleverly across different adapters), trading a small amount of extra compute
for not needing $N$ full model copies.

### 1.6 QLoRA, briefly

QLoRA combines LoRA with quantizing the frozen base weights $W$ to a low-precision format (e.g.
4-bit), while keeping the small trainable $A, B$ matrices in higher precision. Since $W$ is
frozen anyway (never receives gradient updates), quantizing it doesn't hurt training the way
quantizing something you need exact gradients through normally would — this is what makes
fine-tuning very large models feasible on limited GPU memory. Worth mentioning as the natural
next question after "what's LoRA," but out of scope to implement here.


In [ ]:

import torch
import torch.nn as nn

torch.manual_seed(0)



## 2. Implementation

Implement `LoRALinear`, wrapping a frozen base `nn.Linear`. Fill in every `# TODO`.

- `__init__(self, base_linear, r, alpha=None)`: freeze `base_linear`'s parameters, store
  `scaling = alpha / r` (default `alpha = r` if not given), create `A` (shape `(r, in_features)`,
  small random init) and `B` (shape `(out_features, r)`, **zero** init).
- `forward(x)`: `base(x) + scaling * (x @ A^T @ B^T)`.
- `merge(self) -> nn.Linear`: return a plain `nn.Linear` whose weight is
  `base.weight + scaling * (B @ A)` (and copies over the base's bias, if any).


In [ ]:

class LoRALinear(nn.Module):
    def __init__(self, base_linear: nn.Linear, r: int, alpha: float = None):
        super().__init__()
        self.base = base_linear
        # TODO: freeze every parameter of self.base (requires_grad = False)

        in_features = base_linear.in_features
        out_features = base_linear.out_features
        self.r = r
        self.alpha = alpha if alpha is not None else r
        self.scaling = self.alpha / self.r

        # TODO: self.A = nn.Parameter of shape (r, in_features), small random init
        #       (e.g. torch.randn(...) * 0.01)
        # TODO: self.B = nn.Parameter of shape (out_features, r), ZERO init
        self.A = None
        self.B = None

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # TODO: base_out = self.base(x)
        # TODO: delta = x @ A^T @ B^T   (shape: (..., out_features))
        # TODO: return base_out + self.scaling * delta
        raise NotImplementedError

    def merge(self) -> nn.Linear:
        # TODO: build a plain nn.Linear(in_features, out_features, bias=(base has bias)),
        # set its weight to self.base.weight + self.scaling * (B @ A), copy bias if present,
        # return it. Use torch.no_grad() for the in-place parameter copies.
        raise NotImplementedError



### Sanity tests

1. **Zero-init no-op**: immediately after construction (before any training), `LoRALinear`'s
   output must exactly equal the frozen base layer's output alone — this is the whole point of
   zero-initializing $B$.
2. **Merge correctness**: after training $A, B$ for a few steps, `merge()`'s plain `nn.Linear`
   must produce *exactly* the same output as the unmerged `LoRALinear.forward` — if these
   disagree, your merge formula doesn't match your forward formula.
3. **Parameter-count sanity check**: confirm the LoRA parameter count is much smaller than the
   full matrix it's replacing, for a realistic choice of $r$.


In [ ]:

in_f, out_f, r = 16, 12, 4
base = nn.Linear(in_f, out_f)
lora = LoRALinear(base, r=r, alpha=8)

x = torch.randn(5, in_f)

out_lora = lora(x)
out_base = base(x)
assert torch.allclose(out_lora, out_base, atol=1e-6), \
    f"max diff {((out_lora - out_base).abs().max()).item()} -- did you zero-init B?"
print("Test 1 passed: zero-init B makes LoRALinear identical to the frozen base at init")


In [ ]:

opt = torch.optim.SGD([lora.A, lora.B], lr=0.1)
target = torch.randn(5, out_f)
for _ in range(20):
    opt.zero_grad()
    pred = lora(x)
    loss = ((pred - target) ** 2).mean()
    loss.backward()
    opt.step()

out_unmerged = lora(x)
merged_linear = lora.merge()
out_merged = merged_linear(x)
max_diff = (out_unmerged - out_merged).abs().max().item()
assert torch.allclose(out_unmerged, out_merged, atol=1e-5), f"merge mismatch, max diff={max_diff}"
print(f"Test 2 passed: merge() output matches unmerged forward exactly (max diff {max_diff:.2e})")


In [ ]:

lora_params = r * in_f + r * out_f
full_params = in_f * out_f
assert lora_params < full_params
print(f"Test 3 passed: LoRA params = {lora_params} vs. full matrix params = {full_params} "
      f"({100*lora_params/full_params:.1f}% of the full matrix, at r={r})")

assert not lora.base.weight.requires_grad, "base weight should be frozen"
assert lora.A.requires_grad and lora.B.requires_grad, "A and B should be trainable"
print("Confirmed: base is frozen, only A and B are trainable")



## 3. Quiz

1. Why is $B$ zero-initialized rather than $A$ (or both, or neither)? What would go wrong with
   each of those alternatives?
2. After merging, does a LoRA-adapted model cost anything extra at inference time versus the
   unmodified base model? What does merging cost you in flexibility?
3. Why is serving many different LoRA adapters against one shared base model harder than serving
   one merged adapter, and what's the standard way production systems handle it?
4. Why does LoRA tend to get applied to attention projection matrices ($W_Q$, $W_V$, ...) more
   than to the (often much larger) FFN matrices?
5. What determines the right rank $r$ to pick, and what do you give up by choosing $r$ too small
   or gain (and pay for) by choosing it too large?
6. What does QLoRA add on top of LoRA, and why does the fact that $W$ is frozen make it
   *specifically* safe to quantize (versus quantizing weights that are still being trained)?
7. Structurally, how does LoRA's $\frac{\alpha}{r}BA$ update relate to notebook 1's
   Flamingo gated cross-attention $\tanh(\alpha_{gate})$ trick? What's the shared underlying
   principle?

*(Your answers here)*



## 4. Final Answers & Explanations

### Q1 — Why zero-init B specifically
Zero-initializing $B$ (while $A$ gets a small random init) guarantees $BA = 0$ at the very start
of training, so the adapted model is *exactly* the pretrained base model on step zero — training
then gradually moves the model away from that known-good starting point rather than starting
from an immediate random perturbation. If instead $A$ were zero-initialized (and $B$ random),
you'd get the same $BA=0$ result algebraically — this actually also works and is sometimes seen
— but the conventional choice (zero $B$, random $A$) ensures the *gradient* with respect to $A$
is nonzero from the first step (since $\partial \mathcal{L}/\partial A$ depends on $B$, which
would be zero if $B$ were the zero-initialized one, stalling $A$'s gradient at exactly zero on
step one). Zero-initializing *both* would leave the whole update permanently stuck at zero (no
gradient flows to either matrix, since both partial derivatives involve the other, zero, matrix)
— an unrecoverable dead start. Random-initializing *both* would lose the "identical to base
model at $t=0$" guarantee entirely, immediately perturbing the pretrained model with an
untrained, arbitrary low-rank update.

### Q2 — Inference cost after merging
No extra cost whatsoever — `merge()` produces a plain `nn.Linear` with the adaptation baked
directly into its weight matrix, so inference is bit-for-bit the same computation (same shapes,
same single matmul) as the unmodified base model. What you give up is *flexibility*: a merged
model is committed to one specific adapter permanently baked in; if you want a different
adapter, or the base model unmodified, you need a different merged copy (or to keep the
adaptation unmerged and pay the small extra compute of the LoRA path per request, trading
inference simplicity for adapter flexibility).

### Q3 — Multi-adapter serving
Merging bakes exactly one adapter into the shared weights, so serving $N$ different adapters by
merging would require storing $N$ full merged copies of the (potentially huge) base model —
defeating LoRA's entire memory-saving premise. Production multi-adapter serving instead keeps
the base model unmerged and shared across all requests, and applies each request's *specific*
small $A,B$ pair as a lightweight extra computation per request (often batched so different
requests' different adapters can still be processed efficiently together) — paying a small
amount of extra compute per request rather than duplicating the entire base model per adapter.

### Q4 — Why attention projections
Empirically, adapting the attention mechanism's query/value projections captures most of the
behavioral change needed to specialize a pretrained model to a new task, at far lower parameter
cost than adapting the (typically much wider) FFN matrices, since attention projections are
usually smaller matrices to begin with and changes there directly reshape what the model
attends to — often the most task-relevant lever. This is an empirical/practical default from
the LoRA literature, not a theoretical guarantee — many modern recipes do apply LoRA more
broadly (including FFN layers) when the extra parameter/compute budget is available and the
downstream task benefits from it, so it's worth stating as "common practice for cost reasons,"
not an absolute rule.

### Q5 — Choosing rank $r$
Rank $r$ controls how expressive the low-rank update $BA$ can be: too small an $r$ may not have
enough capacity to represent the actual (higher effective rank) change the task needs, capping
achievable fine-tuning quality; too large an $r$ increases trainable parameters and compute
(though still typically far below full fine-tuning) with diminishing returns once $r$ comfortably
exceeds the task's actual intrinsic rank — past that point you're paying more without gaining
more adaptation quality. In practice $r$ is chosen empirically (small values like 4-64 are common
starting points) and tuned like any other hyperparameter, often together with $\alpha$ (since
$\alpha/r$ jointly sets the effective update scale).

### Q6 — QLoRA and why quantizing frozen $W$ is safe
QLoRA quantizes the frozen base weights $W$ (commonly to 4-bit) while keeping the small trainable
$A,B$ matrices at higher precision, and computes the frozen-path forward pass through the
quantized weights. This is specifically safe *because* $W$ never receives a gradient update —
quantization error in $W$ is a fixed, static approximation of a matrix that's never going to
change during this fine-tuning run, so there's no risk of quantization noise compounding through
repeated gradient-based updates the way it would if you tried to quantize weights that were
still being actively trained (where quantization error could interact badly with an evolving
optimization trajectory, e.g. gradients computed through a rounding operation, vanishing updates
smaller than the quantization step, etc.). Freezing $W$ turns "quantize this matrix" into a
one-time, static approximation problem rather than an online training-stability problem.

### Q7 — Connection to Flamingo's gated cross-attention
Both are instances of the same underlying principle: **when splicing a new, randomly-initialized
component into an already-good pretrained model, force that component's contribution to be
exactly zero at initialization** (via a zero-initialized parameter — $B$ here, the gate scalar
$\alpha$ in Flamingo's $\tanh(\alpha)$), so training starts from the pretrained model's known-good
behavior and only gradually introduces the new component's influence as training finds it useful.
The mechanism differs (LoRA: an additive low-rank weight delta; Flamingo: a multiplicative gate
scaling an entire new layer's output) but the motivation and the "guaranteed no-op at $t=0$"
property are identical — this is a general pattern worth recognizing anywhere a new trainable
piece is added on top of a frozen, already-competent pretrained network.
